# 03 - ETL de limpieza (Silver)

## Objetivo
Limpiar y normalizar los 4 tipos de taxi a un esquema canónico, escribiendo el resultado en `clean_data/` (capa Silver). El manejo de valores nulos sigue la siguiente metodología: **primero explorar** (cuántos nulos, en qué columnas, qué % representan) y **después decidir** (eliminar/conservar/reemplazar), nunca al revés.

**Prerrequisitos:**
- `02_perfilamiento_esquemas.ipynb` ya corrido (usamos su resultado, `schema_profile/schema_profile.json`, en vez de repetir el perfilamiento).
- Spark con acceso a S3 vía `s3a://`.

**Salida esperada:** Parquet limpio en `s3://xideralaws-curso-proyecto-alan/clean_data/{tipo}/year={year}/month={month}/`, con columnas canónicas (`pickup_datetime`, `dropoff_datetime`, `PULocationID`, `DOLocationID`, y `distance`/`fare`/`tip` donde aplique) más columnas derivadas (`taxi_type`, `pickup_date`, `pickup_hour`, `day_of_week`, `is_weekend`, `duration_minutes`).

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as sf
import boto3
import json

BUCKET = "xideralaws-curso-proyecto-alan"

config = {
    "spark.jars.packages": "org.apache.hadoop:hadoop-aws:3.4.2,software.amazon.awssdk:bundle:2.29.52",
    "spark.hadoop.fs.s3a.aws.credentials.provider": "software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider",
    "spark.hadoop.fs.s3a.endpoint.region": "us-west-1",
    "spark.driver.memory": "2g"
}
spark = SparkSession.builder.appName("etlLimpiezaSpark").config(map=config).getOrCreate()

:: loading settings :: url = jar:file:/home/ubuntu/opt/spark-4.1.2-bin-hadoop3/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ubuntu/.ivy2.5.2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4ca1aa31-10c8-4db9-af86-83182cf0fcb7;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
:: resolution report :: resolve 406ms :: artifacts dl 19ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.4.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.1.4.Final from central in [default]
	software.amazon.awssdk#bundl

### Reutilizar el perfilamiento de esquemas
En vez de volver a escribir `mapeo_canonico`/`mapeo_economico` a mano, los cargamos del JSON que generó `02_perfilamiento_esquemas.ipynb` — así los dos notebooks quedan realmente conectados, no son copias independientes que se puedan desincronizar.

In [3]:
s3_client = boto3.client("s3", region_name="us-west-1")

respuesta = s3_client.get_object(Bucket=BUCKET, Key="schema_profile/schema_profile.json")
perfil_esquemas = json.loads(respuesta["Body"].read())

mapeo_canonico = perfil_esquemas["columnas_core"]
mapeo_economico = perfil_esquemas["columnas_economicas"]

taxi_types = list(mapeo_canonico.keys())
columnas_core = ["pickup_datetime", "dropoff_datetime", "PULocationID", "DOLocationID"]

print("Mapeo cargado desde schema_profile.json para:", taxi_types)

Mapeo cargado desde schema_profile.json para: ['yellow', 'green', 'fhv', 'fhvhv']


### Perfilamiento de nulos (explorar antes de decidir)
Para cada tipo, contamos filas totales y nulos por columna canónica en **una sola consulta agregada** (no en un loop fila por fila): Spark procesa esto de forma distribuida sin materializar las filas en memoria del driver, así que es seguro incluso sobre el histórico completo de cada tipo.

De paso calculamos la mediana de las columnas económicas con `approxQuantile` (aproximada, no exacta, pero suficiente y mucho más barata computacionalmente) — la usaremos si la decisión más adelante es imputar en vez de eliminar.

In [5]:
perfil_nulos = {}
medianas = {}

for tipo in taxi_types:
    path = f"s3a://{BUCKET}/raw_data/*/*/{tipo}_tripdata_*.parquet"

    columnas_reales = dict(mapeo_canonico[tipo])
    if tipo in mapeo_economico:
        columnas_reales.update(mapeo_economico[tipo])

    seleccion = [sf.col(nombre_real).alias(canonico) for canonico, nombre_real in columnas_reales.items()]
    df = spark.read.parquet(path).select(*seleccion)

    agregados = df.select(
        sf.count(sf.lit(1)).alias("_total"),
        *[sf.sum(sf.col(c).isNull().cast("int")).alias(c) for c in columnas_reales.keys()]
    ).collect()[0].asDict()

    total = agregados.pop("_total")
    perfil_nulos[tipo] = {"total_filas": total, "nulos": agregados}

    print(f"--- {tipo} (total filas: {total}) ---")
    for c, n in agregados.items():
        pct = (n / total * 100) if total else 0
        print(f"  {c}: {n} nulos ({pct:.2f}%)")

    if tipo in mapeo_economico:
        columnas_num = list(mapeo_economico[tipo].keys())
        cuantiles = df.approxQuantile(columnas_num, [0.5], 0.01)
        medianas[tipo] = {c: q[0] for c, q in zip(columnas_num, cuantiles)}
        print(f"  medianas: {medianas[tipo]}")
    print()

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/18 10:16:20 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://xideralaws-curso-proyecto-alan/raw_data/*/*/yellow_tripdata_*.parquet.
java.io.FileNotFoundException: No such file or directory: s3a://xideralaws-curso-proyecto-alan/raw_data/*/*/yellow_tripdata_*.parquet
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:4169)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:4027)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$getFileStatus$21(S3AFileSystem.java:4004)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.invokeTrackingDuration(IOStatisticsBinding.java:547)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDura

--- yellow (total filas: 108891604) ---
  pickup_datetime: 0 nulos (0.00%)
  dropoff_datetime: 0 nulos (0.00%)
  PULocationID: 0 nulos (0.00%)
  DOLocationID: 0 nulos (0.00%)
  distance: 0 nulos (0.00%)
  fare: 0 nulos (0.00%)
  tip: 0 nulos (0.00%)


26/09/18 10:18:06 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://xideralaws-curso-proyecto-alan/raw_data/*/*/green_tripdata_*.parquet.
java.io.FileNotFoundException: No such file or directory: s3a://xideralaws-curso-proyecto-alan/raw_data/*/*/green_tripdata_*.parquet
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:4169)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:4027)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$getFileStatus$21(S3AFileSystem.java:4004)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.invokeTrackingDuration(IOStatisticsBinding.java:547)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5(IOStatisticsBinding.java:528)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDuration(IOStatisticsBinding.java:449)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.trackDurationAnd

  medianas: {'distance': 1.8, 'fare': 14.2, 'tip': 2.3}



--- green (total filas: 1501761) ---
  pickup_datetime: 0 nulos (0.00%)
  dropoff_datetime: 0 nulos (0.00%)
  PULocationID: 0 nulos (0.00%)
  DOLocationID: 0 nulos (0.00%)
  distance: 0 nulos (0.00%)
  fare: 0 nulos (0.00%)
  tip: 0 nulos (0.00%)


...

26/09/18 10:18:24 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://xideralaws-curso-proyecto-alan/raw_data/*/*/fhvhv_tripdata_*.parquet.
java.io.FileNotFoundException: No such file or directory: s3a://xideralaws-curso-proyecto-alan/raw_data/*/*/fhvhv_tripdata_*.parquet
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:4169)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:4027)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$getFileStatus$21(S3AFileSystem.java:4004)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.invokeTrackingDuration(IOStatisticsBinding.java:547)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5(IOStatisticsBinding.java:528)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDuration(IOStatisticsBinding.java:449)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.trackDurationAnd

--- fhv (total filas: 51054441) ---
  pickup_datetime: 0 nulos (0.00%)
  dropoff_datetime: 0 nulos (0.00%)
  PULocationID: 41830310 nulos (81.93%)
  DOLocationID: 8362711 nulos (16.38%)



--- fhvhv (total filas: 609832114) ---
  pickup_datetime: 0 nulos (0.00%)
  dropoff_datetime: 0 nulos (0.00%)
  PULocationID: 0 nulos (0.00%)
  DOLocationID: 0 nulos (0.00%)
  distance: 0 nulos (0.00%)
  fare: 0 nulos (0.00%)
  tip: 0 nulos (0.00%)


[Stage 22:=====================================================>(117 + 1) / 118]

  medianas: {'distance': 2.94, 'fare': 19.5, 'tip': 0.0}



### Decisiones de limpieza (basadas en los números medidos arriba, no en supuestos)
Regla aplicada:
- **Columnas clave** (`pickup_datetime`, `dropoff_datetime`, `PULocationID`, `DOLocationID`): si son nulas, se **elimina la fila** — sin estos campos la fila no sirve para ningún join ni análisis, y el % esperado aquí es cercano a 0.
- **Columnas económicas** (`distance`, `fare`, `tip`):
  - `< 2%` de nulos → **eliminar** (impacto bajo, no vale la pena imputar).
  - `2%-15%` de nulos → **imputar con la mediana** (las distribuciones de tarifa/distancia suelen estar sesgadas a la derecha, así que mediana en vez de media, tal como indicó el profesor).
  - `> 15%` de nulos → **no se imputa automáticamente**, queda marcado para revisión manual (alto riesgo de sesgo si se automatiza a ciegas).

También tratamos los valores **negativos** en columnas económicas (tarifas/distancias negativas no tienen sentido físico) como si fueran nulos, y les aplicamos la misma regla — es un problema de calidad de datos distinto al de los nulos, pero con la misma disciplina de decisión.

In [6]:
def decidir_tratamiento(pct_nulos):
    if pct_nulos < 2:
        return "eliminar"
    elif pct_nulos < 15:
        return "imputar_mediana"
    else:
        return "revisar_manualmente"

UMBRAL_ELIMINAR_UBICACION = 5  # % máximo de nulos en PU/DOLocationID para poder eliminar sin sesgar

decisiones = {}
for tipo in taxi_types:
    decisiones[tipo] = {}
    total = perfil_nulos[tipo]["total_filas"]
    for c, n in perfil_nulos[tipo]["nulos"].items():
        pct = (n / total * 100) if total else 0
        if c in ("pickup_datetime", "dropoff_datetime"):
            decision = "eliminar"
        elif c in ("PULocationID", "DOLocationID"):
            decision = "eliminar" if pct < UMBRAL_ELIMINAR_UBICACION else "conservar_centinela"
        else:
            decision = decidir_tratamiento(pct)
        decisiones[tipo][c] = decision
        print(f"{tipo}.{c}: {pct:.2f}% nulos -> {decision}")

yellow.pickup_datetime: 0.00% nulos -> eliminar
yellow.dropoff_datetime: 0.00% nulos -> eliminar
yellow.PULocationID: 0.00% nulos -> eliminar
yellow.DOLocationID: 0.00% nulos -> eliminar
yellow.distance: 0.00% nulos -> eliminar
yellow.fare: 0.00% nulos -> eliminar
yellow.tip: 0.00% nulos -> eliminar
green.pickup_datetime: 0.00% nulos -> eliminar
green.dropoff_datetime: 0.00% nulos -> eliminar
green.PULocationID: 0.00% nulos -> eliminar
green.DOLocationID: 0.00% nulos -> eliminar
green.distance: 0.00% nulos -> eliminar
green.fare: 0.00% nulos -> eliminar
green.tip: 0.00% nulos -> eliminar
fhv.pickup_datetime: 0.00% nulos -> eliminar
fhv.dropoff_datetime: 0.00% nulos -> eliminar
fhv.PULocationID: 81.93% nulos -> conservar_centinela
fhv.DOLocationID: 16.38% nulos -> conservar_centinela
fhvhv.pickup_datetime: 0.00% nulos -> eliminar
fhvhv.dropoff_datetime: 0.00% nulos -> eliminar
fhvhv.PULocationID: 0.00% nulos -> eliminar
fhvhv.DOLocationID: 0.00% nulos -> eliminar
fhvhv.distance: 0.00% n

### Limpieza incremental por tipo / año / mes
Aquí sí procesamos **un archivo a la vez** (no con un patrón wildcard como en el perfilamiento) porque esta etapa transforma y escribe datos reales, no solo cuenta metadata; es la parte que la t2.medium no puede hacer toda junta de golpe.

Un punto importante es que, al terminar cada iteración, Spark libera esa partición de memoria automáticamente en vez de ir acumulando datos de meses anteriores, por eso no implementamos `cache`. 

Cada escritura va a una ruta explícita (`clean_data/{tipo}/year={year}/month={month}/`) con `mode("overwrite")`, así que el proceso es re-ejecutable: si se corta a la mitad, puedes volver a correrlo sin duplicar nada.

In [8]:
def limpiar_particion(df_raw, tipo):
    columnas_reales = dict(mapeo_canonico[tipo])

    if tipo in mapeo_economico:
        columnas_reales.update(mapeo_economico[tipo])

    seleccion = [
        sf.col(nombre_real).alias(canonico)
        for canonico, nombre_real in columnas_reales.items()
    ]

    df = df_raw.select(*seleccion)

    # Tratar negativos en columnas económicas como nulos
    # (no tienen sentido físico)
    if tipo in mapeo_economico:
        for c in mapeo_economico[tipo].keys():
            df = df.withColumn(
                c,
                sf.when(sf.col(c) < 0, None).otherwise(sf.col(c))
            )

    # Aplicar la decisión medida arriba, columna por columna
    for c, decision in decisiones[tipo].items():

        if decision == "eliminar":
            df = df.filter(sf.col(c).isNotNull())

        elif decision == "imputar_mediana":
            df = df.fillna({c: medianas[tipo][c]})

        elif decision == "conservar_centinela":
            df = df.fillna({c: -1})

        # "revisar_manualmente": se deja tal cual

    df = (
        df
        .withColumn("taxi_type", sf.lit(tipo))
        .withColumn("pickup_date", sf.to_date("pickup_datetime"))
        .withColumn("pickup_hour", sf.hour("pickup_datetime"))
        .withColumn("day_of_week", sf.date_format("pickup_datetime", "E"))
        .withColumn(
            "is_weekend",
            sf.dayofweek("pickup_datetime").isin([1, 7])
        )
        .withColumn(
            "duration_minutes",
            (
                sf.unix_timestamp("dropoff_datetime")
                - sf.unix_timestamp("pickup_datetime")
            ) / 60.0
        )
    )

    return df


paginator = s3_client.get_paginator("list_objects_v2")

for tipo in taxi_types:
    print(f"\n=== Procesando {tipo} ===")

    for page in paginator.paginate(
        Bucket=BUCKET,
        Prefix="raw_data/"
    ):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            filename = key.split("/")[-1]

            if (
                not filename.startswith(f"{tipo}_tripdata_")
                or not filename.endswith(".parquet")
            ):
                continue

            parts = key.split("/")
            year, month = parts[1], parts[2]

            df_raw = spark.read.parquet(
                f"s3a://{BUCKET}/{key}"
            )

            df_limpio = limpiar_particion(
                df_raw,
                tipo
            )

            destino = (
                f"s3a://{BUCKET}/"
                f"clean_data/{tipo}/"
                f"year={year}/month={month}/"
            )

            df_limpio.write.mode("overwrite").parquet(destino)

            print(
                f"  OK {tipo} {year}-{month} -> {destino}"
            )


=== Procesando yellow ===


  OK yellow 2024-01 -> s3a://xideralaws-curso-proyecto-alan/clean_data/yellow/year=2024/month=01/


  OK yellow 2024-02 -> s3a://xideralaws-curso-proyecto-alan/clean_data/yellow/year=2024/month=02/


  OK yellow 2024-03 -> s3a://xideralaws-curso-proyecto-alan/clean_data/yellow/year=2024/month=03/


  OK yellow 2024-04 -> s3a://xideralaws-curso-proyecto-alan/clean_data/yellow/year=2024/month=04/


...

  OK fhvhv 2026-02 -> s3a://xideralaws-curso-proyecto-alan/clean_data/fhvhv/year=2026/month=02/


  OK fhvhv 2026-03 -> s3a://xideralaws-curso-proyecto-alan/clean_data/fhvhv/year=2026/month=03/


  OK fhvhv 2026-04 -> s3a://xideralaws-curso-proyecto-alan/clean_data/fhvhv/year=2026/month=04/


  OK fhvhv 2026-05 -> s3a://xideralaws-curso-proyecto-alan/clean_data/fhvhv/year=2026/month=05/


  OK fhvhv 2026-06 -> s3a://xideralaws-curso-proyecto-alan/clean_data/fhvhv/year=2026/month=06/


### Verificación
Cuenta cuántas particiones quedaron escritas en `clean_data/`, el resultado esperado es obtener una por cada archivo original en `raw_data/` (117, menos las filas que se descartaron por completo si algún archivo terminó vacío tras la limpieza).

In [9]:
respuesta = s3_client.list_objects_v2(Bucket=BUCKET, Prefix="clean_data/", Delimiter="/")
for tipo in taxi_types:
    conteo = 0
    for page in paginator.paginate(Bucket=BUCKET, Prefix=f"clean_data/{tipo}/"):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(".parquet"):
                conteo += 1
    print(f"{tipo}: {conteo} archivos parquet en clean_data/")

yellow: 58 archivos parquet en clean_data/
green: 30 archivos parquet en clean_data/
fhv: 56 archivos parquet en clean_data/
fhvhv: 120 archivos parquet en clean_data/


### Liberar memoria
Cerramos la SparkSession antes de abrir `04_joins_agregaciones_spark.ipynb`.

In [10]:
spark.stop()